# Basic ML Model Deployment

## Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.preprocessing import StandardScaler
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

## Fetch Data

In [2]:

data=pd.read_csv('Loan_data_ver2.csv')

## Explore Data

In [3]:
data.shape

(614, 6)

In [4]:
data.dtypes

Married             object
Education           object
ApplicantIncome      int64
LoanAmount         float64
Credit_History     float64
Loan_Status        float64
dtype: object

In [5]:
data.head(2)

,Married,Education,ApplicantIncome,LoanAmount,Credit_History,Loan_Status
0,No,Graduate,5849,NaN,1.0,0.10
1,Yes,Graduate,4583,128.0,1.0,0.32


In [6]:
# fetch features with missing values
data.isnull().sum()

Married             3
Education           0
ApplicantIncome     0
LoanAmount         22
Credit_History     50
Loan_Status         0
dtype: int64

3 features namely - Married,LoanAmount,Credit_History has missing values

In [7]:
data['Married'].value_counts()

Yes    398
No     213
Name: Married, dtype: int64

In [8]:
data['Education'].value_counts()

Graduate        449
Not Graduate    127
HSC              38
Name: Education, dtype: int64

In [9]:
# segreegating target & feature
X=data.drop('Loan_Status', axis=1)
y=data['Loan_Status']

In [10]:
# spliting data into train & validation set
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=48)

In [11]:
# fetching numeric features list
feat_num=list(X.select_dtypes(include=np.number).columns)


In [12]:
# fetching categorical features  list
feat_cat=list(X.select_dtypes(exclude=np.number).columns)

In [13]:
feat_cat

['Married', 'Education']

## Defining Data processing & Modeling  Pipeline

In [14]:
#  pipeline for numeric atures -missing values replacement using k-Nearest Neighbors follwed by StandardScaler() 
num_pipe=Pipeline([('imputer',KNNImputer()),('std_scale',StandardScaler())])



In [15]:
# pipeline for categorical faetures - missing category replacement by new category i.e. missing followed by one hot encoding 
feat_pipe = Pipeline([('imputer',SimpleImputer(strategy='constant', fill_value='Missing')), 
                      ('one_hot',(OneHotEncoder()))]) 



In [16]:
#combine data processing pipeline
data_pipeline=ColumnTransformer([('numeric',num_pipe,feat_num),
                                 ('categorical',feat_pipe, feat_cat)],
                                remainder='passthrough')



In [17]:
data_pipeline

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer', KNNImputer()),
                                                 ('std_scale',
                                                  StandardScaler())]),
                                 ['ApplicantIncome', 'LoanAmount',
                                  'Credit_History']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('one_hot', OneHotEncoder())]),
                                 ['Married', 'Education'])])

In [18]:
# adding ml-model into pipeline 
full_pipe=Pipeline([('pre_process',data_pipeline),('model',RandomForestRegressor())])

In [19]:
# training
full_pipe.fit(X_train,y_train)

Pipeline(steps=[('pre_process',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('std_scale',
                                                                   StandardScaler())]),
                                                  ['ApplicantIncome',
                                                   'LoanAmount',
                                                   'Credit_History']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 strategy='constant')),
                                                                  ('one_hot',
                                                                   OneHotEncoder())]),
                                                  ['Married', 'Education'])])),
                ('model', RandomForestRegressor())])

In [20]:
# prediction
full_pipe.predict(X_test)

array([0.1564, 0.1844, 0.5378, 0.3768, 0.1457, 0.1043, 0.9602, 0.2519,
       0.2759, 0.264 , 0.1938, 0.9752, 0.3216, 0.1337, 0.0456, 0.1144,
       0.0975, 0.0455, 0.047 , 0.2644, 0.3145, 0.0744, 0.3394, 0.4345,
       0.5913, 0.1476, 0.2389, 0.2115, 0.2242, 0.0095, 0.0966, 0.1617,
       0.3979, 0.9415, 0.1121, 0.2739, 0.2242, 0.5619, 0.9513, 0.1761,
       0.2715, 0.1308, 0.3091, 0.3373, 0.468 , 0.5219, 0.1396, 0.1562,
       0.2711, 0.1757, 0.1064, 0.3899, 0.0688, 0.105 , 0.9616, 0.0093,
       0.397 , 0.291 , 0.2829, 0.209 , 0.3014, 0.857 , 0.2647, 0.1312,
       0.4395, 0.2318, 0.0909, 0.2826, 0.1947, 0.2234, 0.4409, 0.1576,
       0.3503, 0.2901, 0.0673, 0.7687, 0.1056, 0.5101, 0.5983, 0.037 ,
       0.3046, 0.0954, 0.3615, 0.7391, 0.4599, 0.1279, 0.1922, 0.0792,
       0.0572, 0.4771, 0.0587, 0.1828, 0.2389, 0.5698, 0.4007, 0.2174,
       0.9691, 0.0876, 0.0804, 0.3578, 0.4973, 0.0785, 0.0064, 0.0416,
       0.0908, 0.0696, 0.3176, 0.185 , 0.0557, 0.1187, 0.1508, 0.1044,
      

In [21]:
## can store numeric and categorical variables also as pickle file
pickle.dump(feat_num,open('feat_numv1','wb'))
pickle.dump(feat_cat,open('feat_catv1','wb'))

 

## Store the model as pickle file 

In [22]:
pickle.dump(full_pipe,open('full_pipeline','wb'))